# Dysarthria Detection from Audio using Deep Learning Techniques

## Imports

In [ ]:
import os

import librosa
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Reshape, LSTM, Flatten, Layer, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model, Model
from tensorflow.keras import backend as K
from keras_tuner import Hyperband
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split

## Reading and Converting Data to MFCCs

In [ ]:
SR = 16000
N_MFCC = 13
N_FFT = 1400
HOP_LENGTH = 160
SEGMENT_FRAMES = 320

MFCC_ROOT = "mfcc"


def extract_mfcc(y, sr):
    return librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )


def segment_mfcc(mfcc, segment_frames):
    segments = []
    total_frames = mfcc.shape[1]

    for start in range(0, total_frames, segment_frames):
        end = start + segment_frames
        segment = mfcc[:, start:end]

        if segment.shape[1] < segment_frames:
            pad_width = segment_frames - segment.shape[1]
            segment = np.pad(
                segment,
                pad_width=((0, 0), (0, pad_width)),
                mode="constant"
            )

        segments.append(segment)

    return np.array(segments)


def process_and_save(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if not filename.lower().endswith(".wav"):
            continue

        wav_path = os.path.join(input_folder, filename)

        y, sr = librosa.load(wav_path, sr=SR)
        mfcc = extract_mfcc(y, sr)
        mfcc_segments = segment_mfcc(mfcc, SEGMENT_FRAMES)

        save_name = filename.replace(".wav", ".npy")
        save_path = os.path.join(output_folder, save_name)

        np.save(save_path, mfcc_segments)


def load_mfcc_folder(folder, label):
    X, y = [], []

    for filename in os.listdir(folder):
        if not filename.lower().endswith(".npy"):
            continue

        file_path = os.path.join(folder, filename)
        mfcc_segments = np.load(file_path)   # (segments, 13, 320)

        for seg in mfcc_segments:
            X.append(seg)
            y.append(label)

    return np.array(X), np.array(y)

In [ ]:
process_and_save(
    input_folder="data/dysarthria",
    output_folder="mfcc/dysarthria"
)

process_and_save(
    input_folder="data/non_dysarthria",
    output_folder="mfcc/non_dysarthria"
)

### Load MFCCs

In [ ]:
X_dys, y_dys = load_mfcc_folder("mfcc/dysarthria", label=1)
X_non, y_non = load_mfcc_folder("mfcc/non_dysarthria", label=0)
X = np.concatenate((X_dys, X_non), axis=0)
y = np.concatenate((y_dys, y_non), axis=0)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
)

## Preparing Data for Deep Learning Model

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

## Deep Learning Modeling and Evaluation

### ANN Model

In [ ]:
X_train_ann = X_train.reshape(2284, 13 * 320)
X_test_ann = X_test.reshape(572, 13 * 320)

In [ ]:
def build_model(hp):
    model = Sequential()

    model.add(Input(shape=(13 * 320,)))

    for i in range(hp.Int('num_layers', 1, 5)):

        model.add(Dense(hp.Int(f'units_{i}', min_value=64, max_value=512, step=32), activation='relu'))
    
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [ ]:
tuner = Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=5,
    factor=3,
    directory='tuners',
    project_name='ann_tuners'
)

tuner.search(X_train_ann, y_train, validation_data=(X_test_ann, y_test), epochs=5)

In [ ]:
ann = tuner.get_best_models(num_models=1)[0]
ann.summary()

In [ ]:
history = ann.fit(X_train_ann, y_train, epochs=15, validation_data=(X_test_ann, y_test))

In [ ]:
ann.save("models/ann.keras")

### Model Evaluation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(history.history['loss'], label='Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('Loss and Validation Loss over Epochs')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.set_ylim(0, 60)
ax1.set_xlim(0, 15)
ax1.set_xticks(np.arange(0, 15, 1))
ax1.legend()

ax2.plot(history.history['accuracy'], label='Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_title('Accuracy and Validation Accuracy over Epochs')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1)
ax2.set_xlim(0, 15)
ax2.set_xticks(np.arange(0, 15, 1))
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('models/ann_accuracy_loss_plots.png')

In [ ]:
ann = load_model("models/ann.keras")

y_train_pred = (ann.predict(X_train_ann) > 0.5).astype("int32")
y_test_pred = (ann.predict(X_test_ann) > 0.5).astype("int32")

ann_train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
ann_train_accuracy = round(accuracy_score(y_train, y_train_pred), 2)
ann_train_recall = round(recall_score(y_train, y_train_pred), 2)
ann_train_precision = round(precision_score(y_train, y_train_pred), 2)
ann_train_f1 = round(f1_score(y_train, y_train_pred), 2)

ann_test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
ann_test_accuracy = round(accuracy_score(y_test, y_test_pred), 2)
ann_test_recall = round(recall_score(y_test, y_test_pred), 2)
ann_test_precision = round(precision_score(y_test, y_test_pred), 2)
ann_test_f1 = round(f1_score(y_test, y_test_pred), 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(ann_train_confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Train Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(ann_test_confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Test Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('models/ann_confusion_matrix.png')

### CNN Model

In [ ]:
def build_model(hp):
    model = Sequential()
    
    model.add(Input(shape=(13, 320, 1)))

    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(Conv2D(
            filters=hp.Int('conv_1_filters', min_value=8, max_value=24, step=4),
            padding='same',
            kernel_size=(3, 3),
            activation='relu',
        ))
        model.add(MaxPooling2D((2, 2)))

    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [ ]:
tuner = Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=5,
    factor=3,
    directory='tuners',
    project_name='cnn_tuners'
)

tuner.search(X_train, y_train, validation_data=(X_test, y_test))

In [ ]:
cnn = tuner.get_best_models(num_models=1)[0]
cnn.summary()

In [ ]:
history = cnn.fit(X_train, y_train, epochs=15, validation_data=(X_test, y_test))

In [ ]:
cnn.save("models/cnn.keras")

### Model Evaluation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(history.history['loss'], label='Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('Loss and Validation Loss over Epochs')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.set_ylim(0, 40)
ax1.set_xlim(0, 15)
ax1.set_xticks(np.arange(0, 15, 1))
ax1.legend()

ax2.plot(history.history['accuracy'], label='Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_title('Accuracy and Validation Accuracy over Epochs')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1)
ax2.set_xlim(0, 15)
ax2.set_xticks(np.arange(0, 15, 1))
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('models/cnn_accuracy_loss_plots.png')

In [ ]:
cnn = load_model("models/cnn.keras")

y_train_pred = (cnn.predict(X_train) > 0.5).astype("int32")
y_test_pred = (cnn.predict(X_test) > 0.5).astype("int32")

cnn_train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
cnn_train_accuracy = round(accuracy_score(y_train, y_train_pred), 2)
cnn_train_recall = round(recall_score(y_train, y_train_pred), 2)
cnn_train_precision = round(precision_score(y_train, y_train_pred), 2)
cnn_train_f1 = round(f1_score(y_train, y_train_pred), 2)

cnn_test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
cnn_test_accuracy = round(accuracy_score(y_test, y_test_pred), 2)
cnn_test_recall = round(recall_score(y_test, y_test_pred), 2)
cnn_test_precision = round(precision_score(y_test, y_test_pred), 2)
cnn_test_f1 = round(f1_score(y_test, y_test_pred), 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(cnn_train_confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Train Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cnn_test_confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Test Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('models/cnn_confusion_matrix.png')

### CNN-LSTM Model

In [ ]:
def build_model(hp):
    layer_count = 0
    model = Sequential()
    
    model.add(Input(shape=(13, 320, 1)))

    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(Conv2D(
            hp.Int(f'filters_{i}', min_value=8, max_value=32, step=4),
            (3, 3),
            padding='same',
            activation='relu'
        ))
        layer_count += 1
    
    model.add(Reshape((-1, 320)))
    layer_count += 1

    for i in range(hp.Int('num_layers', 1, 2)):
        model.add(LSTM(
            hp.Int(f'lstm_units_{layer_count + i}', min_value=8, max_value=32, step=4),
            return_sequences=True
        ))
        layer_count += 1
    
    model.add(LSTM(
        hp.Int(f'lstm_units{layer_count}', min_value=8, max_value=32, step=4),
        return_sequences=False
    ))

    model.add(Dense(128, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2)),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
tuner = Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=5,
    factor=3,
    directory='tuners',
    project_name='hybrid_tuners'
)

tuner.search(X_train, y_train, validation_data=(X_test, y_test))

In [ ]:
hybrid = tuner.get_best_models(num_models=1)[0]
hybrid.summary()

In [ ]:
history = hybrid.fit(X_train, y_train, epochs=15, validation_data=(X_test, y_test))

In [ ]:
hybrid.save("models/hybrid.keras")

### Model Evaluation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(history.history['loss'], label='Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('Loss and Validation Loss over Epochs')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.set_ylim(0, 5)
ax1.set_xlim(0, 15)
ax1.set_xticks(np.arange(0, 15, 1))
ax1.legend()

ax2.plot(history.history['accuracy'], label='Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_title('Accuracy and Validation Accuracy over Epochs')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.1)
ax2.set_xlim(0, 15)
ax2.set_xticks(np.arange(0, 15, 1))
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('models/hybrid_accuracy_loss_plots.png')

In [ ]:
hybrid = load_model("models/hybrid.keras")

y_train_pred = (hybrid.predict(X_train) > 0.5).astype("int32")
y_test_pred = (hybrid.predict(X_test) > 0.5).astype("int32")

hybrid_train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
hybrid_train_accuracy = round(accuracy_score(y_train, y_train_pred), 2)
hybrid_train_recall = round(recall_score(y_train, y_train_pred), 2)
hybrid_train_precision = round(precision_score(y_train, y_train_pred), 2)
hybrid_train_f1 = round(f1_score(y_train, y_train_pred), 2)

hybrid_test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
hybrid_test_accuracy = round(accuracy_score(y_test, y_test_pred), 2)
hybrid_test_recall = round(recall_score(y_test, y_test_pred), 2)
hybrid_test_precision = round(precision_score(y_test, y_test_pred), 2)
hybrid_test_f1 = round(f1_score(y_test, y_test_pred), 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(hybrid_train_confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Train Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(hybrid_test_confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Test Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('models/hybrid_confusion_matrix.png')

## CapsNet

### Squashing Function

In [ ]:
def squash(vectors, axis=-1):
    s_squared_norm = K.sum(K.square(vectors), axis, keepdims=True)
    scale = s_squared_norm / (1 + s_squared_norm) / K.sqrt(s_squared_norm + K.epsilon())
    return scale * vectors

### Primary Caps

In [ ]:
def PrimaryCaps(inputs, dim_capsule, n_channels, kernel_size, strides, padding):
    x = Conv2D(
        filters=dim_capsule * n_channels,
        kernel_size=kernel_size,
        strides=strides,
        padding=padding,
        activation='relu'
    )(inputs)

    x = Reshape(target_shape=[-1, dim_capsule])(x)
    return Lambda(squash)(x)

### Class Capsule and Dynamic Routing

In [ ]:
class CapsuleLayer(Layer):
    def __init__(self, num_capsules, dim_capsule, routing_iters=3, **kwargs):
        super().__init__(**kwargs)
        self.num_capsules = num_capsules
        self.dim_capsule = dim_capsule
        self.routing_iters = routing_iters

    def build(self, input_shape):
        self.input_num_capsules = input_shape[1]
        self.input_dim_capsule = input_shape[2]

        self.W = self.add_weight(
            shape=(self.input_num_capsules,
                   self.num_capsules,
                   self.input_dim_capsule,
                   self.dim_capsule),
            initializer="glorot_uniform",
            trainable=True
        )

    def call(self, inputs):
        inputs_expand = tf.expand_dims(inputs, 2)
        inputs_tile = tf.tile(inputs_expand, [1, 1, self.num_capsules, 1])

        u_hat = tf.einsum('bijc,ijcd->bijd', inputs_tile, self.W)

        b = tf.zeros_like(u_hat[..., 0])

        for i in range(self.routing_iters):
            c = tf.nn.softmax(b, axis=2)
            s = tf.reduce_sum(c[..., None] * u_hat, axis=1)
            v = squash(s)

            if i < self.routing_iters - 1:
                b += tf.reduce_sum(u_hat * v[:, None, :, :], axis=-1)

        return v

### Margin Loss

In [ ]:
def margin_loss(y_true, y_pred):
    m_plus = 0.9
    m_minus = 0.1
    lambda_val = 0.5

    L = y_true * K.square(K.maximum(0., m_plus - y_pred)) + lambda_val * (1 - y_true) * K.square(K.maximum(0., y_pred - m_minus))

    return K.mean(K.sum(L, axis=1))

### Modeling

In [ ]:
def build_capsnet_model(hp):
    input_shape = (13, 320, 1)
    num_classes = 2

    inputs = Input(shape=input_shape)
    x = inputs

    for i in range(hp.Int('num_layers', min_value=1, max_value=3)):
        x = Conv2D(
            filters=hp.Int(f'conv_{i}_filters', min_value=8, max_value=32, step=8),
            kernel_size=(3, 3),
            padding='same',
            activation='relu'
        )(x)

    primary_caps = PrimaryCaps(
        x,
        dim_capsule=hp.Choice('primary_dim', [4, 8]),
        n_channels=hp.Choice('primary_channels', [8, 16]),
        kernel_size=(3, 3),
        strides=2,
        padding='same'
    )

    class_caps = CapsuleLayer(
        num_capsules=num_classes,
        dim_capsule=hp.Choice('class_dim', [8, 16]),
        routing_iters=hp.Int('routing_iters', 2, 3)
    )(primary_caps)

    outputs = Lambda(
        lambda z: K.sqrt(K.sum(K.square(z), axis=-1)),
        name='capsnet_output'
    )(class_caps)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            hp.Float('lr', 1e-4, 1e-3, sampling='log')
        ),
        loss=margin_loss,
        metrics=['accuracy']
    )

    return model

In [ ]:
tuner = Hyperband(
    build_capsnet_model,
    objective='val_accuracy',
    max_epochs=5,
    factor=3,
    directory='tuners',
    project_name='capsnet_tuners'
)

In [ ]:
tuner.search(
    X_train,
    tf.keras.utils.to_categorical(y_train, 2),
    validation_data=(
        X_test,
        tf.keras.utils.to_categorical(y_test, 2)
    ),
    batch_size=8,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )
    ]
)

### Best Model

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
capsnet = tuner.hypermodel.build(best_hp)
capsnet.summary()

In [ ]:
history = capsnet.fit(
    X_train,
    tf.keras.utils.to_categorical(y_train, 2),
    validation_data=(
        X_test,
        tf.keras.utils.to_categorical(y_test, 2)
    ),
    epochs=20,
    batch_size=8,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )
    ]
)

In [ ]:
capsnet.save("models/capsnet.keras")

### Evaluation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(history.history['loss'], label='Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('Loss and Validation Loss over Epochs')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.set_ylim(0, 5)
ax1.set_xlim(0, 15)
ax1.set_xticks(np.arange(0, 15, 1))
ax1.legend()

ax2.plot(history.history['accuracy'], label='Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_title('Accuracy and Validation Accuracy over Epochs')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.1)
ax2.set_xlim(0, 15)
ax2.set_xticks(np.arange(0, 15, 1))
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('models/capsnet_loss_plots.png')

In [ ]:
y_train_prob = capsnet.predict(X_train)
y_test_prob = capsnet.predict(X_test)

y_train_pred = np.argmax(y_train_prob, axis=1)
y_test_pred = np.argmax(y_test_prob, axis=1)

capsnet_train_confusion_matrix = confusion_matrix(y_train, y_train_pred)
capsnet_train_accuracy = round(accuracy_score(y_train, y_train_pred), 2)
capsnet_train_recall = round(recall_score(y_train, y_train_pred), 2)
capsnet_train_precision = round(precision_score(y_train, y_train_pred), 2)
capsnet_train_f1 = round(f1_score(y_train, y_train_pred), 2)

capsnet_test_confusion_matrix = confusion_matrix(y_test, y_test_pred)
capsnet_test_accuracy = round(accuracy_score(y_test, y_test_pred), 2)
capsnet_test_recall = round(recall_score(y_test, y_test_pred), 2)
capsnet_test_precision = round(precision_score(y_test, y_test_pred), 2)
capsnet_test_f1 = round(f1_score(y_test, y_test_pred), 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(capsnet_train_confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Train Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(capsnet_test_confusion_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Test Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('models/capsnet_confusion_matrix.png')

## Comparison

In [ ]:
import pandas as pd


data = {
    'Model': ['ANN', 'CNN', 'Hybrid', 'CapsNet'],
    'Train Accuracy': [ann_train_accuracy, cnn_train_accuracy, hybrid_train_accuracy, capsnet_train_accuracy],
    'Test Accuracy': [ann_test_accuracy, cnn_test_accuracy, hybrid_test_accuracy, capsnet_test_accuracy],
    'Train Precision': [ann_train_precision, cnn_train_precision, hybrid_train_precision, capsnet_train_precision],
    'Test Precision': [ann_test_precision, cnn_test_precision, hybrid_test_precision, capsnet_test_precision],
    'Train Recall': [ann_train_recall, cnn_train_recall, hybrid_train_recall, capsnet_train_recall],
    'Test Recall': [ann_test_recall, cnn_test_recall, hybrid_test_recall, capsnet_test_recall],
    'Train F1 Score': [ann_train_f1, cnn_train_f1, hybrid_train_f1, capsnet_train_f1],
    'Test F1 Score': [ann_test_f1, cnn_test_f1, hybrid_test_f1, capsnet_test_f1]
}


df = pd.DataFrame(data)
df